In [1]:
# Importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import keras

In [ ]:
# Loading the preprocessed images and labels from .npy files
images = np.load('images.npy')
labels = np.load('labels.npy')

In [3]:
print("Shape of images:", len(images))
print("Shape of labels:", len(labels))

Shape of images: 16011
Shape of labels: 16011


In [ ]:
# Analyzing the unique labels
unique_labels = np.unique(labels)
print("Unique labels:", unique_labels)
print("Number of unique labels:", len(unique_labels))

Unique labels: ['Tomato_Bacterial_spot' 'Tomato_Early_blight' 'Tomato_Late_blight'
 'Tomato_Leaf_Mold' 'Tomato_Septoria_leaf_spot'
 'Tomato_Spider_mites_Two_spotted_spider_mite' 'Tomato__Target_Spot'
 'Tomato__Tomato_YellowLeaf__Curl_Virus' 'Tomato__Tomato_mosaic_virus'
 'Tomato_healthy']
Number of unique labels: 10


In [ ]:
# Preparing the data for training
X = images
y = labels

# Encoding the labels to numerical values
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Converting the labels to one-hot encoding that is suitable for multi-class classification
y = keras.utils.to_categorical(y, num_classes=len(unique_labels))

#Splitting the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Importing the necessary Keras modules for building the CNN model
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
# Definig the CNN model architecture
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dropout(0.25))
model.add(Dense(128, activation='relu'))
model.add(Dense(len(unique_labels), activation='softmax'))

c:\Users\karan\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,840,266 (56.61 MB)

 Trainable params: 14,840,266 (56.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compiling the model with Adam optimizer and categorical crossentropy loss function
model.compile(optimizer='adam', loss=keras.losses.CategoricalCrossentropy(), metrics=['accuracy'])

In [10]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

#Early stopping to prevent overfitting and save the best model based on validation accuracy
early_stopping = EarlyStopping(monitor='val_accuracy', patience=4, verbose=1, min_delta=0.001)

#callback to save the best model based on validation accuracy
model_checkpoint = ModelCheckpoint('./best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)

cb = [early_stopping, model_checkpoint]

In [ ]:
# Training the model with the training data and validating on a validation split of 20%
history = model.fit(X_train, y_train, epochs=30, validation_split=0.2, callbacks=cb)

Epoch 1/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4783 - loss: 1.5493
Epoch 1: val_accuracy improved from None to 0.68644, saving model to ./best_model.h5



Epoch 1: finished saving model to ./best_model.h5
281/281 ━━━━━━━━━━━━━━━━━━━━ 422s 1s/step - accuracy: 0.6522 - loss: 1.0375 - val_accuracy: 0.6864 - val_loss: 0.9442
Epoch 2/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8355 - loss: 0.4835
Epoch 2: val_accuracy improved from 0.68644 to 0.80464, saving model to ./best_model.h5



Epoch 2: finished saving model to ./best_model.h5
281/281 ━━━━━━━━━━━━━━━━━━━━ 522s 2s/step - accuracy: 0.8545 - loss: 0.4217 - val_accuracy: 0.8046 - val_loss: 0.5549
Epoch 3/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9054 - loss: 0.2731
Epoch 3: val_accuracy improved from 0.80464 to 0.88626, saving model to ./best_model.h5



Epoch 3: finished saving model to ./best_model.h5
281/281 ━━━━━━━━━━━━━━━━━━━━ 449s 2s/step - accuracy: 0.9063 - loss: 0.2756 - val_accuracy: 0.8863 - val_loss: 0.3475
Epoch 4/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9443 - loss: 0.1621
Epoch 4: val_accuracy did not improve from 0.88626
281/281 ━━━━━━━━━━━━━━━━━━━━ 450s 2s/step - accuracy: 0.9374 - loss: 0.1833 - val_accuracy: 0.8564 - val_loss: 0.4459
Epoch 5/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9602 - loss: 0.1156
Epoch 5: val_accuracy did not improve from 0.88626
281/281 ━━━━━━━━━━━━━━━━━━━━ 455s 2s/step - accuracy: 0.9568 - loss: 0.1247 - val_accuracy: 0.8483 - val_loss: 0.5328
Epoch 6/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9641 - loss: 0.1050
Epoch 6: val_accuracy improved from 0.88626 to 0.89563, saving model to ./best_model.h5



Epoch 6: finished saving model to ./best_model.h5
281/281 ━━━━━━━━━━━━━━━━━━━━ 426s 2s/step - accuracy: 0.9700 - loss: 0.0890 - val_accuracy: 0.8956 - val_loss: 0.3747
Epoch 7/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9731 - loss: 0.0794
Epoch 7: val_accuracy did not improve from 0.89563
281/281 ━━━━━━━━━━━━━━━━━━━━ 416s 1s/step - accuracy: 0.9663 - loss: 0.1058 - val_accuracy: 0.8577 - val_loss: 0.4380
Epoch 8/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9788 - loss: 0.0692
Epoch 8: val_accuracy did not improve from 0.89563
281/281 ━━━━━━━━━━━━━━━━━━━━ 431s 2s/step - accuracy: 0.9824 - loss: 0.0537 - val_accuracy: 0.8898 - val_loss: 0.4361
Epoch 9/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9811 - loss: 0.0572
Epoch 9: val_accuracy improved from 0.89563 to 0.89920, saving model to ./best_model.h5



Epoch 9: finished saving model to ./best_model.h5
281/281 ━━━━━━━━━━━━━━━━━━━━ 436s 2s/step - accuracy: 0.9826 - loss: 0.0540 - val_accuracy: 0.8992 - val_loss: 0.4185
Epoch 10/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9911 - loss: 0.0308
Epoch 10: val_accuracy did not improve from 0.89920
281/281 ━━━━━━━━━━━━━━━━━━━━ 422s 1s/step - accuracy: 0.9902 - loss: 0.0324 - val_accuracy: 0.8925 - val_loss: 0.4738
Epoch 11/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9874 - loss: 0.0419
Epoch 11: val_accuracy did not improve from 0.89920
281/281 ━━━━━━━━━━━━━━━━━━━━ 414s 1s/step - accuracy: 0.9817 - loss: 0.0594 - val_accuracy: 0.8916 - val_loss: 0.5017
Epoch 12/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9860 - loss: 0.0432
Epoch 12: val_accuracy did not improve from 0.89920
281/281 ━━━━━━━━━━━━━━━━━━━━ 415s 1s/step - accuracy: 0.9829 - loss: 0.0520 - val_accuracy: 0.8760 - val_loss: 0.5572
Epoch 13/30
281/281 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy:

In [ ]:
# Loding the best saved model and evaluating its performance on the test set
model_save = keras.models.load_model('./best_model.h5')

# Evaluating the model on the test set
score = model_save.evaluate(X_test, y_test, verbose=0)

print(f'model accuracy is {score[1]*100}%')

model accuracy is 89.63363766670227%
